# EDA - Base Analítica

**Objetivo:** compreender os fatores associados à alfabetização e à proficiência dos alunos, usando dados do INEP e variáveis de contexto escolar, municipal e estadual.

## 1. Google Colab e Google Drive

Execute esta célula e autorize o acesso ao Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/Othercomputers/@lua2026/01_projects/2026_FIAP/desafios/postech-challenge-3')
sys.path.insert(0, str(PROJECT_ROOT))

!pip install -q loguru python-dotenv pyarrow

## 2. Bibliotecas e leitura do arquivo Parquet

Mantemos o mesmo caminho e a mesma estratégia de leitura do projeto original. O helper `read_parquet` é usado porque a base está particionada por ano.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import PROCESSED_DATA_DIR
from src.preprocessing.io import read_parquet

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

In [ ]:
df = read_parquet(PROCESSED_DATA_DIR / 'base_analitica')

# Metadado técnico do pipeline: não será usado na análise.
if '_gold_processed_at' in df.columns:
    df = df.drop(columns='_gold_processed_at')

print(f'Tamanho da base (linhas, colunas): {df.shape}')
print(f'Anos disponíveis: {sorted(df.ano.dropna().unique())}')
df.head()

## 3. Conhecendo a estrutura da base

Nesta etapa verificamos tipos de dados, nomes das variáveis, valores ausentes e duplicidades. Isso evita usar uma coluna inadequada no modelo.

In [ ]:
df.info()

print('\nQuantidade de valores nulos em toda a base:', df.isnull().sum().sum())
print('Duplicatas na chave ano + id_aluno:', df.duplicated(['ano', 'id_aluno']).sum())

In [ ]:
resumo_colunas = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'valores_unicos': df.nunique(),
    'nulos_%': (df.isnull().mean() * 100).round(2)
}).sort_values('nulos_%', ascending=False)

resumo_colunas.head(20)

## 4. Variáveis que serão analisadas

O alvo principal de classificação é `label_alfabetizado`. A variável `label_proficiencia` pode ser usada em uma futura regressão. Identificadores como `id_aluno` não devem entrar como preditores, pois apenas identificam registros.

In [ ]:
alvo_classificacao = 'label_alfabetizado'
alvo_regressao = 'label_proficiencia'

# Colunas de contexto criadas no pipeline.
variaveis_contexto = [col for col in df.columns if col.startswith('ctx_')]

print('Quantidade de variáveis de contexto:', len(variaveis_contexto))
print('Exemplos:', variaveis_contexto[:10])
print('\nVariáveis de identificação:', ['ano', 'id_aluno', 'id_escola', 'id_municipio', 'id_uf'])

## 5. Estatística descritiva

A descrição mostra média, dispersão e limites das variáveis numéricas. Como a base é grande, começamos pelas variáveis mais diretamente ligadas ao problema.

In [ ]:
colunas_resumo = [col for col in [alvo_regressao, alvo_classificacao, 'ano', 'dependencia_administrativa'] if col in df.columns]
df[colunas_resumo].describe().round(2)

## 6. Distribuição do alvo: alunos alfabetizados

Antes de modelar, verificamos se as classes estão balanceadas. Se uma classe for muito menor que a outra, a acurácia sozinha não será uma métrica suficiente.

In [ ]:
distribuicao_alvo = df[alvo_classificacao].value_counts(dropna=False).sort_index()
percentual_alvo = (df[alvo_classificacao].value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)

print('Contagem:')
display(distribuicao_alvo)
print('\nPercentual:')
display(percentual_alvo)

sns.countplot(data=df, x=alvo_classificacao, color='#4C78A8')
plt.title('Distribuição do alvo: alfabetizado')
plt.xlabel('Label alfabetizado')
plt.ylabel('Quantidade de alunos')
plt.show()

## 7. Comportamento ao longo dos anos

Uma diferença importante entre anos pode indicar mudança de contexto, de coleta ou de desempenho. Essa análise também orienta uma validação temporal do modelo.

In [ ]:
resultado_por_ano = df.groupby('ano').agg(
    alunos=('id_aluno', 'size'),
    taxa_alfabetizacao=(alvo_classificacao, 'mean'),
    proficiencia_media=(alvo_regressao, 'mean')
).reset_index()

resultado_por_ano['taxa_alfabetizacao'] = (resultado_por_ano['taxa_alfabetizacao'] * 100).round(2)
resultado_por_ano['proficiencia_media'] = resultado_por_ano['proficiencia_media'].round(2)
resultado_por_ano

In [ ]:
sns.barplot(data=resultado_por_ano, x='ano', y='taxa_alfabetizacao', color='#59A14F')
plt.title('Taxa de alfabetização por ano')
plt.xlabel('Ano')
plt.ylabel('Taxa de alfabetização (%)')
plt.show()

## 8. Análise por dependência administrativa

Esta comparação ajuda a identificar padrões entre redes de ensino. Ela descreve associações, mas não prova que a dependência administrativa seja a causa do resultado.

In [ ]:
analise_dependencia = df.groupby('dependencia_administrativa').agg(
    alunos=('id_aluno', 'size'),
    taxa_alfabetizacao=(alvo_classificacao, 'mean'),
    proficiencia_media=(alvo_regressao, 'mean')
).reset_index()

analise_dependencia['taxa_alfabetizacao'] = (analise_dependencia['taxa_alfabetizacao'] * 100).round(2)
analise_dependencia['proficiencia_media'] = analise_dependencia['proficiencia_media'].round(2)
analise_dependencia

In [ ]:
sns.barplot(data=analise_dependencia, x='dependencia_administrativa', y='taxa_alfabetizacao', color='#F28E2B')
plt.title('Taxa de alfabetização por dependência administrativa')
plt.xlabel('Dependência administrativa')
plt.ylabel('Taxa de alfabetização (%)')
plt.show()

## 9. Distribuição e outliers da proficiência

O boxplot permite observar concentração, dispersão e valores extremos. Valores extremos não devem ser removidos automaticamente: primeiro é preciso verificar se são erros ou casos reais.

In [ ]:
# Amostra apenas para acelerar os gráficos; as tabelas anteriores usam a base inteira.
amostra = df.sample(n=min(100000, len(df)), random_state=42)

sns.histplot(data=amostra, x=alvo_regressao, bins=40, kde=True, color='#4C78A8')
plt.title('Distribuição da proficiência (amostra)')
plt.xlabel('Proficiência')
plt.show()

sns.boxplot(data=amostra, y=alvo_regressao, color='#76B7B2')
plt.title('Boxplot da proficiência (amostra)')
plt.ylabel('Proficiência')
plt.show()

## 10. Correlações entre variáveis numéricas

A correlação mede associação linear. Correlações muito altas entre preditores podem indicar multicolinearidade, o que é especialmente relevante para modelos lineares.

In [ ]:
# Selecionamos até 12 variáveis numéricas de contexto para o gráfico permanecer legível.
numericas_contexto = df[variaveis_contexto].select_dtypes(include='number').columns.tolist()
colunas_correlacao = [alvo_regressao] + numericas_contexto[:12]
correlacao = amostra[colunas_correlacao].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlacao, cmap='rocket', center=0, annot=False)
plt.title('Matriz de correlação: proficiência e variáveis de contexto')
plt.show()

correlacao[alvo_regressao].sort_values(ascending=False)

## 11. Relação entre a variável mais correlacionada e a proficiência

O gráfico abaixo ajuda a verificar visualmente se a relação apontada pela correlação parece consistente. Correlação não significa causalidade.

In [ ]:
correlacoes_alvo = correlacao[alvo_regressao].drop(alvo_regressao).abs().sort_values(ascending=False)
melhor_variavel = correlacoes_alvo.index[0]
print('Variável de contexto mais correlacionada com proficiência:', melhor_variavel)

sns.regplot(data=amostra, x=melhor_variavel, y=alvo_regressao,
            scatter_kws={'alpha': 0.15, 's': 10}, line_kws={'color': 'red'})
plt.title(f'Proficiência x {melhor_variavel} (amostra)')
plt.xlabel(melhor_variavel)
plt.ylabel('Proficiência')
plt.show()

## 12. Conclusões para a modelagem

Preencha esta seção após executar as células anteriores. Sugestão de interpretação:

1. **Qualidade dos dados:** registre colunas com muitos nulos, duplicatas e tipos que precisam ser corrigidos.
2. **Alvo:** informe se `label_alfabetizado` está desbalanceado. Se estiver, avalie `class_weight`, F1-score, recall, precision e ROC-AUC.
3. **Tempo:** se houver diferença entre anos, prefira validar o modelo respeitando a ordem temporal.
4. **Variáveis relevantes:** priorize variáveis com associação consistente ao alvo, mas remova identificadores como `id_aluno`.
5. **Multicolinearidade:** em modelos lineares, considere remover ou agrupar variáveis altamente correlacionadas.
6. **Hipóteses analíticas:** diferenças por ano, dependência administrativa e contexto socioeconômico podem explicar parte da variação da proficiência; elas devem ser testadas no modelo e não assumidas como causalidade.